# Large and wide tables

Column summaries and integrity checks examine every row. Relationship
discovery screens broadly, then limits follow-up work by budget. This lets
the profiler handle tables with millions of rows and hundreds of columns,
while the amount of relationship evidence depends on the configured
budget.

## What stays exact


In [ ]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Pass': ['Column cards', 'Duplicate rows', 'Single-column keys', 'Sortedness', 'Dates', 'Arithmetic identities', 'Functional dependencies'], 'At any size': ['every row: exact cardinality, null counts, statistics, quantiles, top values, concentration', 'full-row hash plus exact verification', 'exact distinct counts over the full column', 'exact scan of adjacent values', 'exact calendar scan of distinct days', 'fitted on a sample, residuals verified on every pair-complete row', 'screened broadly; reported candidates verified over up to 200,000 rows (`standard`) or 500,000 (`deep`) — full data when the table is smaller']})
(GT(_doc_table)
 .fmt_markdown(columns=['Pass', 'At any size'])
 .tab_header(title='What stays exact')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


Dense and nullable Int64/UInt64/Float64 columns borrow Arrow buffers directly;
other dtypes can require a normalized copy. This is an in-memory profiler,
so input size and exact statistics still drive memory and runtime.

## Bounded relationship work

Screening is broad and cheap; follow-up is bounded. The budget table in
[](coverage.ipynb) gives the per-tier numbers: at `standard`, the encoded
relationship sample is 20,000 rows over all usable columns, each target is
screened against all columns and fitted against at most 48 shortlisted
predictors over 5,000 rows, all numeric pairs are screened and the 496
strongest get rank statistics, and FD candidates are verified over 200,000
rows. `deep` raises each of those. Coverage reports the limits used for each run.

Here is a wide synthetic table with
relationships planted in the *last* columns — deliberately at the far end
of the column order, because a screener that only looked at the first few
dozen columns would miss them:


In [ ]:
import numpy as np
import polars as pl

import onceover

rng = np.random.default_rng(20241024)
n, width = 100_000, 80

data = {"id": np.arange(n, dtype=np.int64)}
data.update({f"noise_{i:02}": rng.random(n) for i in range(width - 7)})
x = rng.uniform(-1, 1, n)
codes = rng.integers(0, 4, n, dtype=np.int64)
dirty_id = np.arange(n, dtype=np.int64)
dirty_id[-1] = 0
data.update({
    "late_x": x,
    "late_affine": 3 * x + 2,       # arithmetic identity with late_x
    "late_curve": x * x,            # nonlinear response
    "dirty_id": dirty_id,
    "late_code": codes,             # code -> label dependency
})

df = pl.DataFrame(data).with_columns(
    pl.col("late_code").cast(pl.String).alias("late_label")
)
assert df.shape == (n, width)
print(df.shape)

report = onceover.profile(df)
print(report.overview["headline"])

The report recovers the planted affine dependency, nonlinear response,
and code→label lookup in the final columns, and ranks the integer id as
the key candidate. Screening considers every usable column before
shortlisting.

The coverage block records the work performed:

In [ ]:
from great_tables import GT, loc, style

rows = [{"pass": p, "status": i["status"], "detail": i["detail"]}
        for p, i in report.coverage.items()]

(GT(pl.DataFrame(rows))
 .tab_header(title=f"Coverage on {df.height:,} rows × {df.width} columns (standard)",
             subtitle="Note the caps: the detectors screened everything, then bounded the deep work")
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"),
            loc.body()))

## The scale benchmark

The scale benchmark checks runtime, resident memory, and recovery together.
Its fixed-seed fixture has 2,000,000 rows and 200 columns, including an
integer primary key, a key with one repeated value, and the three
relationships illustrated above. Those relationships occupy columns beyond
position 64. The script fails unless all eight recovery and cardinality
checks pass.

From the repository root, with the checkout's package installed:

```bash
python -m pip install '.[benchmark]'
python benchmarks/scale.py --rows 2000000 --columns 200 --threads 4 \
  --output /tmp/onceover-scale.json
```

The recorded comparison uses release builds, the standard budget, Python
3.13.7, Polars 2.0.0, and four Rayon threads on a macOS arm64 host with
24 GiB of system memory. These are recorded results, not timings of the
smaller demonstration above.


In [ ]:
recorded_scale_results = pl.DataFrame({
    "Measurement": [
        "Profiling wall time", "Peak process RSS", "Increase from pre-profile RSS",
        "Recovery and cardinality checks", "Independent noise targets reported predictable",
    ],
    "0.1.2": ["351.08 s", "5.84 GiB", "2.77 GiB", "8 / 8", "0"],
    "0.1.3": ["79.67 s", "3.47 GiB", "0.40 GiB", "8 / 8", "0"],
})
(GT(recorded_scale_results)
 .tab_header(title="Recorded scale benchmark: 2,000,000 rows × 200 columns",
             subtitle="Standard budget · fixed seed 20241024 · four Rayon threads")
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"),
            loc.body()))


Both versions recovered all planted structures, with no independent noise
target reported as predictable. RSS includes the input and Python process;
the increase from pre-profile RSS is not an allocation count for scratch
space. The 0.1.2 run briefly shared the host with other validation tasks,
while the 0.1.3 run began after those tasks completed. The comparison is a
single-run observation, not a guarantee of a particular speedup.

The [benchmark protocol and records](https://github.com/fcela/onceover/blob/main/docs/benchmarks/large-table-performance.md)
explain the timing boundaries, environment, and interpretation. The older
0.1.1 record predates several default detectors and is not a comparable
runtime baseline.

## What bounded search can miss

Samples can miss rare patterns: a relationship visible in 0.01% of rows may
not appear in a 20,000-row sample. Shortlisting can miss weak signals and
interaction-only signals: a pair of columns that predict a target only
*together* may not survive per-column screening. Capped searches stop
early: beyond 496 pairs (`standard`), the profiler still screens every
numeric pair but computes detailed statistics only for the strongest.

Coverage records sample sizes and search caps, and each finding includes
its validation scope. A run that reports no relationship may still have
missed one.
Increase the budget (`deep`, `sample_rows`, `max_pairs`) or test a
specific hypothesis directly.

```{admonition} Full-data scan costs
:class: note

Bounded *relationship* work does not bound the input: exact column
summaries examine every row, so runtime and memory grow with the table.
The budget tiers bound discovery, not the scan.
```

## Tuning for a big table

The overrides from [](coverage.ipynb) apply directly:

```python
# wider sampling when the standard caps feel tight
report = onceover.profile(df, sample_rows=50_000, max_predictors=96,
                          max_pairs=2_000)

# or a wall-clock budget; coverage lists passes skipped after the deadline
report = onceover.profile(df, deadline_seconds=30.0)
```

Sample and pair overrides change the amount of relationship work. A
deadline can skip later passes. Neither reduces the rows examined by
full-data passes that complete.
